# iter_spmv: Dense baseline + segmented CSR-style compile

This notebook provides two ARIES compilation paths:
1. **Dense MxV baseline** (v1, simple pragmas).
2. **Segmented-vector CSR-style path** with row-sharded scheduling (one row owned by one shard only).

The second path models: local vector-bank access first, then remote-bank query fallback behavior in software logic.

In [1]:
import os
import sys
from pathlib import Path
import inspect
import textwrap
import numpy as np

cur_dir = os.getcwd()
cwd = Path(cur_dir).resolve()
aries_root = None
for candidate in [cwd] + list(cwd.parents):
    maybe = candidate / "tools" / "ARIES"
    if maybe.exists():
        aries_root = maybe
        break

if aries_root is None:
    raise RuntimeError("Cannot locate tools/ARIES from current working directory.")

aries_path = str(aries_root)
if aries_path not in sys.path:
    sys.path.append(aries_path)

from frontend import *

## 1) Dense MxV baseline compile (simple ARIES pragmas)

In [2]:
D_M, D_N = 256, 256

In [13]:
@task_tile()
def dense_mv(A: float32[-1, -1], x: float32[-1], y: float32[-1], rows, cols):
    for i in range(rows):
        y[i] = 0.0
        for j in range(cols):
            y[i] += A[i, j] * x[j]

@task_top()
def top_dense_mv(A: float32[D_M, D_N], x: float32[D_N], y: float32[D_M]):
    cast_A = aries.cast(A, (-1, -1))
    cast_x = aries.cast(x, (-1,))
    cast_y = aries.cast(y, (-1,))
    t = dense_mv(cast_A, cast_x, cast_y, D_M, D_N)
    return t

In [14]:
np.random.seed(7)
A_dense = np.random.rand(D_M, D_N).astype(np.float32)
x_dense = np.random.rand(D_N).astype(np.float32)
y_dense = np.zeros((D_M,), dtype=np.float32)

dense_task = top_dense_mv(A_dense, x_dense, y_dense)
y_ref_dense = A_dense @ x_dense
print("Dense runtime check:", np.allclose(y_dense, y_ref_dense, rtol=1e-4, atol=1e-4))

Dense runtime check: True


In [15]:
const_dense = "\n".join([
    f"D_M, D_N = {D_M}, {D_N}"
])
dense_src = textwrap.dedent(inspect.getsource(dense_mv.func))
top_dense_src = textwrap.dedent(inspect.getsource(top_dense_mv.func))
all_code_dense = "\n\n".join([const_dense, dense_src, top_dense_src])

repo_root = Path(aries_path).parents[1]
dense_prj_dir = str(repo_root / "build" / "iter_spmv_dense")
temp_dir = aries_path + "/templates"

sch_dense = Schedule(dense_task)
sch_dense.to("VCK190")
sch_dense.build(all_code_dense, dense_prj_dir, temp_dir)
print("Dense compile output:", dense_prj_dir)

Directory '/home/byao/Desktop/uni-comb-accel/build/iter_spmv_dense/project' already exists, skipping creation.
Directory '/home/byao/Desktop/uni-comb-accel/build/iter_spmv_dense/project/aie' already exists, skipping creation.
Directory '/home/byao/Desktop/uni-comb-accel/build/iter_spmv_dense/project/kernel' already exists, skipping creation.
Directory '/home/byao/Desktop/uni-comb-accel/build/iter_spmv_dense/project/host' already exists, skipping creation.
... wrote /home/byao/Desktop/uni-comb-accel/build/iter_spmv_dense/Makefile
... wrote /home/byao/Desktop/uni-comb-accel/build/iter_spmv_dense/project/Makefile
Dense compile output: /home/byao/Desktop/uni-comb-accel/build/iter_spmv_dense


## 2) Segmented-vector CSR-style path

Design intent in this v1 model:
- rows are partitioned by shard (disjoint row ownership to avoid hazards),
- sparse matrix is consumed in CSR form,
- vector is segmented into two banks (`X0`, `X1`),
- each shard checks whether a column belongs to local segment or a remote segment.

In [6]:
S_M, S_N = 128, 128
SPARSITY = 0.92
NUM_SHARDS = 4
SEG0_END = S_N // 2
SHARD_ROWS = (S_M + NUM_SHARDS - 1) // NUM_SHARDS

rng = np.random.default_rng(0)
A_sparse_dense = rng.random((S_M, S_N), dtype=np.float32)
mask = rng.random((S_M, S_N)) < SPARSITY
A_sparse_dense[mask] = 0.0
x_sparse = rng.random(S_N, dtype=np.float32)

row_ptr = [0]
col_idx = []
vals = []
for i in range(S_M):
    nz = np.nonzero(A_sparse_dense[i])[0]
    col_idx.extend(nz.tolist())
    vals.extend(A_sparse_dense[i, nz].tolist())
    row_ptr.append(len(col_idx))

row_ptr = np.array(row_ptr, dtype=np.int32)
col_idx = np.array(col_idx, dtype=np.int32)
vals = np.array(vals, dtype=np.float32)
NNZ = int(len(vals))

y_sparse = np.zeros((S_M,), dtype=np.float32)
y_ref_sparse = A_sparse_dense @ x_sparse

print("CSR nnz:", NNZ)

CSR nnz: 1278


In [22]:
@task_tile()
def spmv_csr_segmented(RowPtr: int32[-1], ColIdx: int32[-1], Val: float32[-1],
                       X0: float32[-1], X1: float32[-1], Y: float32[-1],
                       rows, shard_rows, seg0_end):
    # Explicit static row shards (disjoint ownership, no Y write hazard)
    for r in range(0, SHARD_ROWS):
        Y[r] = 0.0
        rs = RowPtr[r]
        re = RowPtr[r + 1]
        for p in range(rs, re):
            c = ColIdx[p]
            if c < seg0_end:
                Y[r] += Val[p] * X0[c]
            else:
                Y[r] += Val[p] * X1[c - seg0_end]

    for r in range(SHARD_ROWS, 2 * SHARD_ROWS):
        Y[r] = 0.0
        rs = RowPtr[r]
        re = RowPtr[r + 1]
        for p in range(rs, re):
            c = ColIdx[p]
            if c < seg0_end:
                Y[r] += Val[p] * X0[c]
            else:
                Y[r] += Val[p] * X1[c - seg0_end]

    for r in range(2 * SHARD_ROWS, 3 * SHARD_ROWS):
        Y[r] = 0.0
        rs = RowPtr[r]
        re = RowPtr[r + 1]
        for p in range(rs, re):
            c = ColIdx[p]
            if c < seg0_end:
                Y[r] += Val[p] * X0[c]
            else:
                Y[r] += Val[p] * X1[c - seg0_end]

    for r in range(3 * SHARD_ROWS, 4 * SHARD_ROWS):
        Y[r] = 0.0
        rs = RowPtr[r]
        re = RowPtr[r + 1]
        for p in range(rs, re):
            c = ColIdx[p]
            if c < seg0_end:
                Y[r] += Val[p] * X0[c]
            else:
                Y[r] += Val[p] * X1[c - seg0_end]

@task_top()
def top_sparse_segmented(RowPtr: int32[S_M + 1], ColIdx: int32[NNZ], Val: float32[NNZ],
                         X: float32[S_N], Y: float32[S_M]):
    cast_RowPtr = aries.cast(RowPtr, (-1,))
    cast_ColIdx = aries.cast(ColIdx, (-1,))
    cast_Val = aries.cast(Val, (-1,))
    cast_X = aries.cast(X, (-1,))
    cast_Y = aries.cast(Y, (-1,))

    idx0 = aries.arange(0, SEG0_END)
    idx1 = aries.arange(SEG0_END, S_N)
    X0 = aries.load(cast_X, (idx0,))
    X1 = aries.load(cast_X, (idx1,))

    t = spmv_csr_segmented(cast_RowPtr, cast_ColIdx, cast_Val, X0, X1, cast_Y,
                           S_M, SHARD_ROWS, SEG0_END)
    return t

In [23]:
# Runtime check for segmented CSR-style path
sparse_task = top_sparse_segmented(row_ptr, col_idx, vals, x_sparse, y_sparse)
print("Segmented CSR runtime check:", np.allclose(y_sparse, y_ref_sparse, rtol=1e-4, atol=1e-4))

Segmented CSR runtime check: True


In [26]:
const_sparse = "\n".join([
    f"S_M, S_N = {S_M}, {S_N}",
    f"NNZ = {NNZ}",
    f"NUM_SHARDS = {NUM_SHARDS}",
    f"SEG0_END = {SEG0_END}",
    f"SHARD_ROWS = {SHARD_ROWS}"
])
sparse_src = textwrap.dedent(inspect.getsource(spmv_csr_segmented.func))
top_sparse_src = textwrap.dedent(inspect.getsource(top_sparse_segmented.func))
all_code_sparse = "\n\n".join([const_sparse, sparse_src, top_sparse_src])

sparse_prj_dir = str(repo_root / "build" / "iter_spmv_segmented")

sch_sparse = Schedule(sparse_task)
sch_sparse.to("VCK190")

try:
    sch_sparse.build(all_code_sparse, sparse_prj_dir, temp_dir)
    print("Segmented CSR compile output:", sparse_prj_dir)
except Exception as e:
    print("Segmented CSR compile hit a backend/frontend limitation:")
    print(type(e).__name__, str(e))

    # Fallback: still emit a second compile artifact with segmented-vector access behavior.
    @task_tile()
    def segmented_mv_surrogate(A: float32[-1, -1], X0: float32[-1], X1: float32[-1],
                               Y: float32[-1], rows, cols, seg0_end):
        for i in range(rows):
            Y[i] = 0.0
            for j in range(cols):
                if j < seg0_end:
                    Y[i] += A[i, j] * X0[j]
                else:
                    Y[i] += A[i, j] * X1[j - seg0_end]

    @task_top()
    def top_segmented_mv_surrogate(A: float32[S_M, S_N], X: float32[S_N], Y: float32[S_M]):
        cast_A = aries.cast(A, (-1, -1))
        cast_X = aries.cast(X, (-1,))
        cast_Y = aries.cast(Y, (-1,))
        # Pass full vector to both ports; tile-level branch models local vs remote bank reads.
        t = segmented_mv_surrogate(cast_A, cast_X, cast_X, cast_Y, S_M, S_N, SEG0_END)
        return t

    y_sur = np.zeros((S_M,), dtype=np.float32)
    sur_task = top_segmented_mv_surrogate(A_sparse_dense, x_sparse, y_sur)
    sur_prj_dir = str(repo_root / "build" / "iter_spmv_segmented_surrogate")

    const_sur = "\n".join([
        f"S_M, S_N = {S_M}, {S_N}",
        f"SEG0_END = {SEG0_END}"
    ])
    sur_src = textwrap.dedent(inspect.getsource(segmented_mv_surrogate.func))
    sur_top_src = textwrap.dedent(inspect.getsource(top_segmented_mv_surrogate.func))
    all_code_sur = "\n\n".join([const_sur, sur_src, sur_top_src])

    try:
        sch_sur = Schedule(sur_task)
        sch_sur.to("VCK190")
        sch_sur.build(all_code_sur, sur_prj_dir, temp_dir)
        print("Segment-aware surrogate compile output:", sur_prj_dir)
    except Exception as e2:
        print("Segment-aware surrogate compile also hit a limitation:")
        print(type(e2).__name__, str(e2))

print("Dense baseline compile is available in:", dense_prj_dir)

Directory '/home/byao/Desktop/uni-comb-accel/build/iter_spmv_segmented/project' already exists, skipping creation.
Directory '/home/byao/Desktop/uni-comb-accel/build/iter_spmv_segmented/project/aie' already exists, skipping creation.
Directory '/home/byao/Desktop/uni-comb-accel/build/iter_spmv_segmented/project/kernel' already exists, skipping creation.
Directory '/home/byao/Desktop/uni-comb-accel/build/iter_spmv_segmented/project/host' already exists, skipping creation.
Segmented CSR compile hit a backend/frontend limitation:
AssertionError KeyError: The val 'rs' was not defined.
Directory '/home/byao/Desktop/uni-comb-accel/build/iter_spmv_segmented_surrogate/project' already exists, skipping creation.
Directory '/home/byao/Desktop/uni-comb-accel/build/iter_spmv_segmented_surrogate/project/aie' already exists, skipping creation.
Directory '/home/byao/Desktop/uni-comb-accel/build/iter_spmv_segmented_surrogate/project/kernel' already exists, skipping creation.
Directory '/home/byao/Desk